# Diabetes progression: Ridge regression

Regression baseline on the `diabetes` dataset; metrics are RMSE and R² (`ml_toolkit.metrics.regression_metrics`).

In [1]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = "png"
plt.rcParams["figure.dpi"] = 72
from sklearn.pipeline import make_pipeline
from ml_toolkit.data import make_split
from ml_toolkit.features import make_preprocessor
from ml_toolkit.models import build_model
from ml_toolkit.metrics import regression_metrics

In [2]:
split = make_split("diabetes", seed=7, stratify=False)
split.sizes()

{'train': 264, 'val': 89, 'test': 89}

## Regularisation sweep

In [3]:
rows = []
for alpha in [0.01, 0.1, 1.0, 10.0, 100.0]:
    reg = make_pipeline(make_preprocessor(), build_model("ridge", alpha=alpha)).fit(split.X_train, split.y_train)
    rows.append({"alpha": alpha, **regression_metrics(split.y_val, reg.predict(split.X_val))})
alpha_table = pd.DataFrame(rows).round(3)
alpha_table

,alpha,rmse,r2
0,0.01,53.549,0.514
1,0.10,53.557,0.513
2,1.00,53.631,0.512
3,10.00,53.937,0.507
4,100.00,55.851,0.471


In [4]:
best_alpha = float(alpha_table.sort_values("rmse").iloc[0].alpha)
final = make_pipeline(make_preprocessor(), build_model("ridge", alpha=best_alpha)).fit(split.X_train, split.y_train)
print(f"best alpha = {best_alpha}")
print("TEST:", {k: round(v, 3) for k, v in regression_metrics(split.y_test, final.predict(split.X_test)).items()})

best alpha = 0.01
TEST: {'rmse': 53.438, 'r2': 0.426}


Linear model is a reasonable baseline; next step would be gradient boosting on the same split.